In [ ]:
# ============================================================
# CELL — Check file header (zip files always start with 'PK')
# ============================================================
with open("archive.zip", "rb") as f:
    header = f.read(4)
    print("File header bytes:", header)
    print("Is valid ZIP signature:", header[:2] == b'PK')

File header bytes: b'PK\x03\x04'
Is valid ZIP signature: True


In [ ]:
# ============================================================
# CELL — Check file size and status
# ============================================================
import os

zip_path = "archive.zip"

if os.path.exists(zip_path):
    size_mb = os.path.getsize(zip_path) / (1024*1024)
    print(f"File exists: {zip_path}")
    print(f"Size: {size_mb:.2f} MB")
else:
    print("File does not exist yet — upload may still be in progress")

File exists: archive.zip
Size: 60.32 MB


In [ ]:
# ============================================================
# CELL — Extract FER2013 zip file
# ============================================================
import zipfile
import os

zip_path = "archive.zip"   # adjust if your zip has a different name

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall("fer2013")

print("Extraction done.")
print("Contents:", os.listdir("fer2013"))

Extraction done.
Contents: ['train', 'test']


In [ ]:
# ============================================================
# CELL — Verify folder structure and counts
# ============================================================
import os

TRAIN_DIR = "fer2013/train"
TEST_DIR  = "fer2013/test"

emotion_labels = [
    'angry','disgust','fear',
    'happy','neutral','sad','surprise'
]

print("TRAIN set:")
total_train = 0
for emotion in emotion_labels:
    path  = os.path.join(TRAIN_DIR, emotion)
    count = len(os.listdir(path)) if os.path.exists(path) else 0
    total_train += count
    print(f"  {emotion:10s}: {count} images")
print(f"  TOTAL: {total_train} images\n")

print("TEST set:")
total_test = 0
for emotion in emotion_labels:
    path  = os.path.join(TEST_DIR, emotion)
    count = len(os.listdir(path)) if os.path.exists(path) else 0
    total_test += count
    print(f"  {emotion:10s}: {count} images")
print(f"  TOTAL: {total_test} images")

TRAIN set:
  angry     : 3995 images
  disgust   : 436 images
  fear      : 4097 images
  happy     : 7215 images
  neutral   : 4965 images
  sad       : 4830 images
  surprise  : 3171 images
  TOTAL: 28709 images

TEST set:
  angry     : 958 images
  disgust   : 111 images
  fear      : 1024 images
  happy     : 1774 images
  neutral   : 1233 images
  sad       : 1247 images
  surprise  : 831 images
  TOTAL: 7178 images


In [ ]:
# ============================================================
# CELL — FER2013 Dataset class
# ============================================================
import torch
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import torchvision.transforms as transforms

class FERDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.samples   = []
        self.transform = transform
        self.labels    = emotion_labels

        for idx, emotion in enumerate(self.labels):
            folder = os.path.join(root_dir, emotion)
            if not os.path.exists(folder):
                continue
            for fname in os.listdir(folder):
                if fname.endswith(('.jpg','.png','.jpeg')):
                    self.samples.append(
                        (os.path.join(folder, fname), idx)
                    )

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = Image.open(path).convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img, label

transform_train = transforms.Compose([
    transforms.Resize((48, 48)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize([0.5],[0.5])
])

transform_test = transforms.Compose([
    transforms.Resize((48, 48)),
    transforms.ToTensor(),
    transforms.Normalize([0.5],[0.5])
])

train_fer = FERDataset(TRAIN_DIR, transform=transform_train)
test_fer  = FERDataset(TEST_DIR,  transform=transform_test)

train_fer_loader = DataLoader(train_fer, batch_size=64, shuffle=True)
test_fer_loader  = DataLoader(test_fer,  batch_size=64)

print(f"Train images: {len(train_fer)}")
print(f"Test images:  {len(test_fer)}")

Train images: 28709
Test images:  7178


In [ ]:
# ============================================================
# CELL — Check GPU (should already be True if Colab GPU is on)
# ============================================================
import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Device:", device)

Device: cuda


In [ ]:
# ============================================================
# CELL — Build CNN model for facial emotion
# ============================================================
import torch.nn as nn

class FacialEmotionCNN(nn.Module):
    def __init__(self, num_classes=7):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 6 * 6, 256),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        return self.classifier(self.features(x))

model_fer = FacialEmotionCNN(num_classes=7).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model_fer.parameters(), lr=0.001)

print("FER Model ready.")
print("Total parameters:", sum(p.numel() for p in model_fer.parameters()))

FER Model ready.
Total parameters: 1274951


In [ ]:
# Quick fix — add class weights to loss function
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

# Get all training labels
all_train_labels = [label for _, label in train_fer.samples]

class_weights = compute_class_weight(
    'balanced',
    classes=np.unique(all_train_labels),
    y=all_train_labels
)
class_weights = torch.FloatTensor(class_weights).to(device)

# Use weighted loss instead
criterion = nn.CrossEntropyLoss(weight=class_weights)

In [ ]:
# ============================================================
# CELL — Train FER model
# ============================================================
import time

EPOCHS = 10

for epoch in range(EPOCHS):
    model_fer.train()
    total_loss = 0
    correct    = 0
    total      = 0
    start_time = time.time()

    for images, labels in train_fer_loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model_fer(images)
        loss    = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        preds       = outputs.argmax(dim=1)
        correct    += (preds == labels).sum().item()
        total      += labels.size(0)

    train_acc  = correct / total
    epoch_time = (time.time() - start_time) / 60

    print(f"Epoch {epoch+1}/{EPOCHS} | "
          f"Loss: {total_loss/len(train_fer_loader):.4f} | "
          f"Train Acc: {train_acc:.4f} | "
          f"Time: {epoch_time:.1f} min")

torch.save(model_fer.state_dict(), "fer_model.pt")
print("\nSaved fer_model.pt")

Epoch 1/10 | Loss: 1.0927 | Train Acc: 0.6073 | Time: 0.3 min
Epoch 2/10 | Loss: 1.0048 | Train Acc: 0.6248 | Time: 0.4 min
Epoch 3/10 | Loss: 0.9476 | Train Acc: 0.6408 | Time: 0.3 min
Epoch 4/10 | Loss: 0.9088 | Train Acc: 0.6514 | Time: 0.3 min
Epoch 5/10 | Loss: 0.8811 | Train Acc: 0.6576 | Time: 0.4 min
Epoch 6/10 | Loss: 0.8343 | Train Acc: 0.6731 | Time: 0.3 min
Epoch 7/10 | Loss: 0.8078 | Train Acc: 0.6810 | Time: 0.3 min
Epoch 8/10 | Loss: 0.7812 | Train Acc: 0.6883 | Time: 0.3 min
Epoch 9/10 | Loss: 0.7553 | Train Acc: 0.6958 | Time: 0.3 min
Epoch 10/10 | Loss: 0.7366 | Train Acc: 0.6972 | Time: 0.3 min

Saved fer_model.pt


In [ ]:
# ============================================================
# CELL — Evaluate FER model on test set
# ============================================================
from sklearn.metrics import classification_report, accuracy_score, f1_score

model_fer.eval()
all_preds  = []
all_labels = []

with torch.no_grad():
    for images, labels in test_fer_loader:
        images  = images.to(device)
        outputs = model_fer(images)
        preds   = outputs.argmax(dim=1).cpu()
        all_preds.extend(preds.tolist())
        all_labels.extend(labels.tolist())

print("="*55)
print("FER MODEL — FINAL TEST RESULTS")
print("="*55)
print(f"Accuracy : {accuracy_score(all_labels, all_preds):.4f}")
print(f"F1 Macro : {f1_score(all_labels, all_preds, average='macro'):.4f}")
print()
print(classification_report(
    all_labels, all_preds,
    target_names=emotion_labels
))

FER MODEL — FINAL TEST RESULTS
Accuracy : 0.5879
F1 Macro : 0.5647

              precision    recall  f1-score   support

       angry       0.46      0.57      0.51       958
     disgust       0.42      0.65      0.51       111
        fear       0.41      0.45      0.43      1024
       happy       0.83      0.77      0.80      1774
     neutral       0.54      0.56      0.55      1233
         sad       0.49      0.35      0.41      1247
    surprise       0.72      0.77      0.74       831

    accuracy                           0.59      7178
   macro avg       0.55      0.59      0.56      7178
weighted avg       0.59      0.59      0.59      7178

